# Proyecto final · Predicción de partidos ATP
## Paso 3 · Limpieza

Partimos de las dos tablas que guardó el paso 2:
- `tabla_principal.csv.gz`: una fila por partido del tour, con la etiqueta `gana_A`.
- `historial_partidos.csv.gz`: todos los partidos (tour, qualys y challengers) que usaremos para construir variables en el paso 4.

Cada decisión de limpieza queda justificada con un número, y al final se muestra la tabla **antes y después**.

| # | Problema | Decisión |
|---|---|---|
| 1 | Duplicados | Revisar y eliminar si existen |
| 2 | Retiros y walkovers | Eliminar walkovers; conservar retiros |
| 3 | Superficie Carpet y sin superficie | Carpet → Hard; completar faltantes por torneo |
| 4 | Nivel "O" | Revisar qué es y decidir |
| 5 | Ranking faltante o demasiado viejo | Completar con el ranking reciente; el resto, "sin ranking" |
| 6 | Estatura faltante | Comparar métodos de imputación por su error |
| 7 | Mano, edad, siembra, tipo de entrada | Completar con reglas simples |
| 8 | Estadísticas del partido | Marcar las válidas (solo para el historial) |
| 9 | Columnas con fuga | Separarlas de las de entrada |

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold

SEMILLA = 42
CARPETA = os.environ.get("TENIS_DATOS", "datos")
AB = ["A", "B"]

tabla = pd.read_csv(os.path.join(CARPETA, "tabla_principal.csv.gz"), low_memory=False,
                    parse_dates=["fecha", "A_nacimiento", "B_nacimiento"])
hist = pd.read_csv(os.path.join(CARPETA, "historial_partidos.csv.gz"), low_memory=False,
                   parse_dates=["tourney_date"])
original = tabla.copy()          # para la tabla de antes y después
bitacora = []                    # registro de filas eliminadas

def exactitud_regla(d):
    """Regla del paso 1: gana el de mejor ranking (sin ranking = peor)."""
    rA, rB = d["A_rank"].fillna(np.inf), d["B_rank"].fillna(np.inf)
    pred = np.where(rA < rB, 1, np.where(rA > rB, 0, 1))
    return (pred == d["gana_A"]).mean()

print(f"Tabla principal: {tabla.shape[0]:,} filas × {tabla.shape[1]} columnas")
print(f"Historial: {hist.shape[0]:,} filas × {hist.shape[1]} columnas")
print(f"Exactitud de la regla del ranking antes de limpiar: {exactitud_regla(tabla):.1%}")

Tabla principal: 79,299 filas × 65 columnas
Historial: 304,839 filas × 31 columnas
Exactitud de la regla del ranking antes de limpiar: 65.6%


### 1 · Duplicados
Se revisan tres cosas: el identificador de partido repetido, el mismo partido cargado dos veces (mismos jugadores, torneo y ronda, sin importar quién quedó como A) y partidos de un jugador contra sí mismo.

In [2]:
par = np.sort(tabla[["A_id", "B_id"]].values, axis=1)
clave = (tabla["tourney_id"].astype(str) + "|" + tabla["ronda"].astype(str) + "|"
         + par[:, 0].astype(str) + "|" + par[:, 1].astype(str))

print(f"match_id repetido:            {tabla['match_id'].duplicated().sum()}")
print(f"Mismo partido dos veces:      {clave.duplicated().sum()}")
print(f"Jugador contra sí mismo:      {(tabla['A_id'] == tabla['B_id']).sum()}")

quitar = clave.duplicated() | (tabla["A_id"] == tabla["B_id"])
tabla = tabla[~quitar].reset_index(drop=True)
bitacora.append(("Duplicados o jugador contra sí mismo", int(quitar.sum())))

match_id repetido:            0
Mismo partido dos veces:      2
Jugador contra sí mismo:      0


### 2 · Retiros, walkovers y partidos sin terminar
La columna `score` trae el marcador y, a veces, cómo terminó el partido:
- **W/O (walkover):** el partido **no se jugó**. No hay nada que predecir: se elimina (las casas de apuestas también anulan estas apuestas).
- **RET (retiro):** se jugó, pero un jugador abandonó por lesión. **Se conserva**: antes del partido nadie sabe que habrá un retiro, así que quitarlos haría la evaluación más fácil de lo que es en la realidad.
- **DEF (descalificación):** mismo criterio que el retiro.
- **Sin marcador o no terminado:** no se sabe con certeza quién ganó: se elimina.

In [3]:
def tipo_final(s):
    if pd.isna(s) or str(s).strip() == "":
        return "sin marcador"
    s = str(s).upper()
    if "W/O" in s or "WALKOVER" in s:
        return "walkover"
    if "DEF" in s:
        return "descalificación"
    if "RET" in s:
        return "retiro"
    if any(p in s for p in ["UNFINISHED", "ABN", "UNK", "PROGRESS", "PLAYED"]):
        return "no terminado"
    return "completo"

tabla["tipo_final"] = tabla["score"].map(tipo_final)
resumen_final = (tabla.groupby("tipo_final")
                 .apply(lambda d: pd.Series({"partidos": len(d),
                                             "% del total": round(len(d) / len(tabla) * 100, 2),
                                             "exactitud regla (%)": round(exactitud_regla(d) * 100, 1)}),
                        include_groups=False)
                 .sort_values("partidos", ascending=False))
resumen_final

,partidos,% del total,exactitud regla (%)
tipo_final,,,
completo,76481.0,96.45,66.1
retiro,2364.0,2.98,54.4
walkover,423.0,0.53,56.5
descalificación,20.0,0.03,30.0
no terminado,9.0,0.01,44.4


In [4]:
eliminar = ["walkover", "sin marcador", "no terminado"]
quitar = tabla["tipo_final"].isin(eliminar)
for t in eliminar:
    bitacora.append((f"Partido {t}", int((tabla["tipo_final"] == t).sum())))
tabla = tabla[~quitar].reset_index(drop=True)

# Marca de retiro: es información POSTERIOR al partido, así que nunca entra al modelo
tabla["termino_anormal"] = tabla["tipo_final"].isin(["retiro", "descalificación"]).astype(int)
print(f"Eliminados: {quitar.sum():,}. Quedan {len(tabla):,} partidos, "
      f"de los cuales {tabla['termino_anormal'].sum():,} terminaron por retiro o descalificación.")

# El mismo criterio para el historial: los walkovers no cuentan como partido jugado
hist["tipo_final"] = hist["score"].map(tipo_final)
antes_hist = len(hist)
hist = hist[~hist["tipo_final"].isin(eliminar)].reset_index(drop=True)
print(f"Historial: se eliminan {antes_hist - len(hist):,} partidos no jugados.")

Eliminados: 432. Quedan 78,865 partidos, de los cuales 2,384 terminaron por retiro o descalificación.
Historial: se eliminan 1,389 partidos no jugados.


### 3 · Superficie
- **Carpet:** superficie bajo techo y rápida que el ATP dejó de usar en 2009. En el paso 1 la regla del ranking rindió igual en Carpet (64,2 %) y en Hard (66,2 %), y por velocidad Carpet se parece más a la pista dura bajo techo. Se une con **Hard**, para no dejar una categoría que ya no existe.
- **Sin superficie:** un torneo suele jugarse en la misma superficie todos los años. Se completa con la superficie más frecuente del mismo torneo en otros años; si no se puede, se elimina.

In [5]:
print("Superficie antes:")
print(tabla["superficie"].value_counts(dropna=False).to_string())

base = pd.concat([
    tabla[["torneo", "superficie"]],
    hist[["tourney_name", "surface"]].set_axis(["torneo", "superficie"], axis=1),
]).dropna()
moda_torneo = base.groupby("torneo")["superficie"].agg(lambda s: s.mode().iat[0])

falta = tabla["superficie"].isna()
print(f"\nPartidos sin superficie: {falta.sum()}  →  torneos: {tabla.loc[falta, 'torneo'].unique().tolist()}")
tabla.loc[falta, "superficie"] = tabla.loc[falta, "torneo"].map(moda_torneo)
print(f"Completados con la superficie del mismo torneo: {(falta & tabla['superficie'].notna()).sum()}")

quitar = tabla["superficie"].isna()
bitacora.append(("Sin superficie (no se pudo completar)", int(quitar.sum())))
tabla = tabla[~quitar].reset_index(drop=True)
tabla["superficie"] = tabla["superficie"].replace("Carpet", "Hard")

# Lo mismo en el historial
falta_h = hist["surface"].isna()
hist.loc[falta_h, "surface"] = hist.loc[falta_h, "tourney_name"].map(moda_torneo)
hist["surface"] = hist["surface"].replace("Carpet", "Hard")

print("\nSuperficie después:")
print(tabla["superficie"].value_counts(dropna=False).to_string())

Superficie antes:
superficie
Hard      43192
Clay      25636
Grass      7954
Carpet     2030
NaN          53

Partidos sin superficie: 53  →  torneos: ['Davis Cup WG1 PO: BRA vs CHN', 'Davis Cup WG1 PO: ISR vs LAT', 'Davis Cup WG1 PO: PAK vs LTU', 'Davis Cup WG1 PO: PER vs IRL', 'Davis Cup WG1 PO: ROU vs THA', 'Davis Cup WG1 PO: TUR vs SLO', 'Davis Cup WG1 PO: UKR vs LBN', 'Davis Cup WG2 PO: BAR vs POC', 'Davis Cup WG2 PO: BOL vs GEO', 'Davis Cup WG2 PO: EGY vs PAR', 'Davis Cup WG2 PO: ESA vs JOR', 'Davis Cup WG2 PO: EST vs JAM', 'Davis Cup WG2 PO: INA vs VIE', 'Davis Cup WG2 PO: RSA vs LUX', 'Davis Cup WG2 PO: TUN vs CYP']
Completados con la superficie del mismo torneo: 3

Superficie después:
superficie
Hard     45222
Clay     25636
Grass     7957


### 4 · Nivel del torneo "O"
Veamos qué torneos son antes de decidir.

In [6]:
print(tabla.loc[tabla["nivel"] == "O", ["torneo", "anio", "best_of"]]
      .value_counts().reset_index(name="partidos").to_string(index=False))
print("\nNiveles:", tabla["nivel"].value_counts().to_dict())

        torneo  anio  best_of  partidos
Paris Olympics  2024        3        63

Niveles: {'A': 42617, 'M': 15101, 'G': 13289, 'D': 7280, 'F': 465, 'O': 63}


Si son los **Juegos Olímpicos**, son partidos reales de nivel ATP entre profesionales y se conservan con su propia categoría. Son pocos, así que en el modelo el codificador de categorías debe tolerar valores que no aparezcan en el entrenamiento (`handle_unknown="ignore"`).

### 5 · Ranking faltante o demasiado viejo
En el paso 2, la unión con los rankings semanales encontró ranking para algunos partidos donde el archivo no lo traía. Pero en algunos casos ese ranking tenía años de antigüedad (hasta 7.935 días). Un ranking así no describe al jugador de hoy.

**Regla:** se usa el ranking del archivo. Si falta, se usa el unido **solo si tiene 60 días o menos** (unas 8 semanas). Si no, el jugador queda **sin ranking**.

In [7]:
LIMITE_DIAS = 60
for L in AB:
    falta_archivo = tabla[f"{L}_rank"].isna()
    unido = tabla[f"{L}_rank_unido"].notna()
    reciente = tabla[f"{L}_dias_ranking"] <= LIMITE_DIAS
    print(f"Jugador {L}: sin ranking en el archivo = {falta_archivo.sum():,}")
    print(f"   la unión encontró uno: {(falta_archivo & unido).sum():,}  "
          f"→ reciente (≤ {LIMITE_DIAS} días): {(falta_archivo & unido & reciente).sum():,}, "
          f"viejo: {(falta_archivo & unido & ~reciente).sum():,}")
    print("   antigüedad de esos rankings (días):",
          tabla.loc[falta_archivo & unido, f"{L}_dias_ranking"].quantile([.25, .5, .75, .95]).round(0).to_dict())

    usar = falta_archivo & unido & reciente
    tabla.loc[usar, f"{L}_rank"] = tabla.loc[usar, f"{L}_rank_unido"]
    tabla.loc[usar, f"{L}_rank_points"] = tabla.loc[usar, f"{L}_pts_unido"]

Jugador A: sin ranking en el archivo = 1,025
   la unión encontró uno: 450  → reciente (≤ 60 días): 56, viejo: 394
   antigüedad de esos rankings (días): {0.25: 130.0, 0.5: 417.0, 0.75: 995.0, 0.95: 2281.0}
Jugador B: sin ranking en el archivo = 1,096
   la unión encontró uno: 510  → reciente (≤ 60 días): 42, viejo: 468
   antigüedad de esos rankings (días): {0.25: 172.0, 0.5: 468.0, 0.75: 995.0, 0.95: 2256.0}


Los que siguen sin ranking **no son datos perdidos al azar**: son jugadores que no están clasificados (juveniles, regresos de lesión, invitados). Por eso no se imputan con la media:
- Se crea la marca `sin_ranking` (1 si no tiene ranking).
- El ranking se llena con **2.000**, peor que cualquier clasificado, y los puntos con **0**.

Así el modelo sabe que es "peor que el último clasificado", que es lo que realmente significa.

In [8]:
print(f"Ranking máximo observado: {tabla[['A_rank', 'B_rank']].max().max():,.0f}")
for L in AB:
    tabla[f"{L}_sin_ranking"] = tabla[f"{L}_rank"].isna().astype(int)
    print(f"{L}: quedan sin ranking {tabla[f'{L}_sin_ranking'].sum():,} partidos "
          f"({tabla[f'{L}_sin_ranking'].mean():.2%})")
    tabla[f"{L}_rank"] = tabla[f"{L}_rank"].fillna(2000)
    tabla[f"{L}_rank_points"] = tabla[f"{L}_rank_points"].fillna(0)

Ranking máximo observado: 2,173
A: quedan sin ranking 969 partidos (1.23%)
B: quedan sin ranking 1,054 partidos (1.34%)


### 6 · Estatura faltante: comparar métodos por su error
Igual que con la edad en Titanic, no se elige a ojo: se esconde el 20 % de las estaturas **conocidas**, se imputan con cada método y se mide el error contra el valor real (validación cruzada de 5 particiones, a nivel de jugador).

Métodos: media global, mediana global, mediana por país, mediana por década de nacimiento (los jugadores son cada vez más altos) y mediana por país y década.

In [9]:
# Primero, completar con la tabla de jugadores (paso 2) y revisar valores imposibles
for L in AB:
    tabla[f"{L}_ht"] = tabla[f"{L}_ht"].fillna(tabla[f"{L}_estatura_j"])
    tabla[f"{L}_ioc"] = tabla[f"{L}_ioc"].fillna(tabla[f"{L}_pais_j"])

jug = pd.concat([
    tabla[[f"{L}_id", f"{L}_ht", f"{L}_ioc", f"{L}_nacimiento"]].set_axis(["id", "ht", "ioc", "nacimiento"], axis=1)
    for L in AB
]).drop_duplicates("id").reset_index(drop=True)
jug["decada"] = (jug["nacimiento"].dt.year // 10) * 10

print(f"Jugadores distintos: {len(jug):,}; con estatura: {jug['ht'].notna().sum():,} ({jug['ht'].notna().mean():.1%})")
print(f"Rango de estaturas: {jug['ht'].min():.0f} a {jug['ht'].max():.0f} cm")
raras = jug["ht"].notna() & ~jug["ht"].between(155, 215)
print(f"Estaturas fuera de 155–215 cm (se tratan como faltantes): {raras.sum()}")
jug.loc[raras, "ht"] = np.nan

Jugadores distintos: 2,782; con estatura: 1,654 (59.5%)
Rango de estaturas: 3 a 211 cm
Estaturas fuera de 155–215 cm (se tratan como faltantes): 3


In [10]:
def imputar(train, test, metodo):
    g = train["ht"].median()
    if metodo == "Media global":
        return np.full(len(test), train["ht"].mean())
    if metodo == "Mediana global":
        return np.full(len(test), g)
    if metodo == "Mediana por país":
        return test["ioc"].map(train.groupby("ioc")["ht"].median()).fillna(g).values
    if metodo == "Mediana por década":
        return test["decada"].map(train.groupby("decada")["ht"].median()).fillna(g).values
    if metodo == "Mediana por país y década":
        m2 = train.groupby(["ioc", "decada"])["ht"].median()
        v = pd.Series(m2.reindex(pd.MultiIndex.from_frame(test[["ioc", "decada"]])).values, index=test.index)
        return v.fillna(test["ioc"].map(train.groupby("ioc")["ht"].median())).fillna(g).values

METODOS = ["Media global", "Mediana global", "Mediana por país", "Mediana por década", "Mediana por país y década"]
conocidos = jug[jug["ht"].notna()].reset_index(drop=True)
errores = {m: [] for m in METODOS}
for tr, te in KFold(5, shuffle=True, random_state=SEMILLA).split(conocidos):
    train, test = conocidos.iloc[tr], conocidos.iloc[te]
    for m in METODOS:
        errores[m].append(np.abs(imputar(train, test, m) - test["ht"].values).mean())

tabla_errores = (pd.DataFrame(errores).agg(["mean", "std"]).T
                 .rename(columns={"mean": "Error medio (cm)", "std": "Desv. entre particiones"})
                 .round(2).sort_values("Error medio (cm)"))
tabla_errores

,Error medio (cm),Desv. entre particiones
Mediana por país,4.99,0.22
Mediana global,5.08,0.29
Mediana por década,5.09,0.28
Media global,5.10,0.28
Mediana por país y década,5.33,0.33


In [11]:
mejor = tabla_errores.index[0]
print(f"Método elegido: {mejor} (error medio {tabla_errores.iloc[0, 0]} cm)")

faltan = jug[jug["ht"].isna()]
estimada = pd.Series(imputar(conocidos, faltan, mejor), index=faltan["id"].values)
estatura_final = pd.concat([conocidos.set_index("id")["ht"], estimada])

for L in AB:
    tabla[f"{L}_ht_imputada"] = (~tabla[f"{L}_id"].isin(conocidos["id"])).astype(int)
    tabla[f"{L}_ht"] = tabla[f"{L}_id"].map(estatura_final)
    print(f"{L}: estaturas imputadas en {tabla[f'{L}_ht_imputada'].sum():,} partidos")

Método elegido: Mediana por país (error medio 4.99 cm)
A: estaturas imputadas en 2,279 partidos
B: estaturas imputadas en 2,301 partidos


La estatura no depende del resultado del partido, así que imputarla con la mediana de un grupo no filtra la etiqueta. Se calcula con los jugadores conocidos y se aplica a los que faltan.

### 7 · Mano, edad, siembra y tipo de entrada

In [12]:
# Mano: R (diestro), L (zurdo), U (desconocida)
for L in AB:
    tabla[f"{L}_hand"] = tabla[f"{L}_hand"].fillna(tabla[f"{L}_mano_j"])
print("Mano antes:", pd.concat([tabla["A_hand"], tabla["B_hand"]]).value_counts(dropna=False).to_dict())
for L in AB:
    tabla[f"{L}_hand"] = tabla[f"{L}_hand"].where(tabla[f"{L}_hand"].isin(["R", "L"]), "R")
print("Mano después:", pd.concat([tabla["A_hand"], tabla["B_hand"]]).value_counts().to_dict())
print("→ La desconocida se imputa como diestro, la moda (más del 85 % de los jugadores).\n")

# Edad: si falta, se calcula desde la fecha de nacimiento; si tampoco hay, la mediana
for L in AB:
    falta = tabla[f"{L}_age"].isna()
    tabla.loc[falta, f"{L}_age"] = (tabla.loc[falta, "fecha"] - tabla.loc[falta, f"{L}_nacimiento"]).dt.days / 365.25
    sigue = tabla[f"{L}_age"].isna()
    tabla.loc[sigue, f"{L}_age"] = tabla[f"{L}_age"].median()
    print(f"Edad {L}: faltaban {falta.sum()}, calculadas desde nacimiento {(falta & ~sigue).sum()}, con mediana {sigue.sum()}")

# Siembra: vacío = no fue cabeza de serie. Entrada: vacío = aceptación directa (DA)
for L in AB:
    tabla[f"{L}_seed"] = pd.to_numeric(tabla[f"{L}_seed"], errors="coerce")
    tabla[f"{L}_sembrado"] = tabla[f"{L}_seed"].notna().astype(int)
    tabla[f"{L}_seed"] = tabla[f"{L}_seed"].fillna(0)
    tabla[f"{L}_entry"] = tabla[f"{L}_entry"].fillna("DA").astype(str).str.upper()
print("\nTipo de entrada:", pd.concat([tabla["A_entry"], tabla["B_entry"]]).value_counts().to_dict())
print("(Q = qualy, WC = invitación, LL = lucky loser, PR = ranking protegido, DA = directa)")

Mano antes: {'R': 136354, 'L': 20007, 'U': 1253, 'A': 12, nan: 4}
Mano después: {'R': 137623, 'L': 20007}
→ La desconocida se imputa como diestro, la moda (más del 85 % de los jugadores).

Edad A: faltaban 11, calculadas desde nacimiento 0, con mediana 11
Edad B: faltaban 14, calculadas desde nacimiento 0, con mediana 14

Tipo de entrada: {'DA': 131064, 'Q': 14868, 'WC': 8455, 'LL': 2395, 'PR': 571, 'SE': 197, 'ALT': 54, 'NG': 12, 'ITF': 6, 'W': 4, 'UP': 2, 'L': 1, 'S': 1}
(Q = qualy, WC = invitación, LL = lucky loser, PR = ranking protegido, DA = directa)


La siembra y el tipo de entrada se conocen **antes** del partido (se publican con el cuadro), así que sí pueden entrar al modelo.

### 8 · Estadísticas del partido (solo para el historial)
Las estadísticas (aces, puntos de saque, etc.) no entran al modelo directamente, pero en el paso 4 se usarán para calcular promedios de partidos **anteriores**. Aquí se marca cuáles son confiables:
- que no falten,
- que haya puntos de saque (`svpt` > 0),
- que sean coherentes (primeros saques adentro ≤ puntos de saque; primeros saques ganados ≤ primeros saques adentro).

In [13]:
def stats_validas(d, w="w_", l="l_"):
    ok = pd.Series(True, index=d.index)
    for p in [w, l]:
        cols = [f"{p}{c}" for c in ["svpt", "1stIn", "1stWon", "2ndWon"]]
        ok &= d[cols].notna().all(axis=1)
        ok &= d[f"{p}svpt"] > 0
        ok &= d[f"{p}1stIn"] <= d[f"{p}svpt"]
        ok &= d[f"{p}1stWon"] <= d[f"{p}1stIn"]
    return ok

hist["stats_validas"] = stats_validas(hist).astype(int)
hist["periodo"] = pd.cut(hist["tourney_date"].dt.year, [1994, 1999, 2007, 2015, 2027],
                         labels=["1995-1999", "2000-2007", "2008-2015", "2016-2026"])
(hist.pivot_table(index="periodo", columns="origen", values="stats_validas", aggfunc="mean", observed=True) * 100).round(1)

origen,qual_chall,tour
periodo,,
1995-1999,0.0,89.1
2000-2007,0.0,88.6
2008-2015,70.4,89.3
2016-2026,98.7,96.2


Los partidos sin estadísticas válidas siguen contando para la forma y el Elo (se sabe quién ganó), pero no para los promedios de saque.

### 9 · Columnas con fuga
Estas columnas solo se conocen **después** del partido. Quedan en la tabla para construir el historial, pero **no pueden entrar al modelo**:

In [14]:
ESTADISTICAS = ["ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "SvGms", "bpSaved", "bpFaced"]
FUGA = ["score", "minutos", "tipo_final", "termino_anormal"] + [f"{L}_{c}" for L in AB for c in ESTADISTICAS]
AUXILIARES = [f"{L}_{c}" for L in AB for c in ["rank_unido", "pts_unido", "dias_ranking", "estatura_j", "mano_j", "pais_j"]]

tabla = tabla.drop(columns=AUXILIARES)     # ya se usaron para completar; no se necesitan más
print(f"Columnas con fuga (solo historial): {len(FUGA)}")
print(FUGA)

Columnas con fuga (solo historial): 22
['score', 'minutos', 'tipo_final', 'termino_anormal', 'A_ace', 'A_df', 'A_svpt', 'A_1stIn', 'A_1stWon', 'A_2ndWon', 'A_SvGms', 'A_bpSaved', 'A_bpFaced', 'B_ace', 'B_df', 'B_svpt', 'B_1stIn', 'B_1stWon', 'B_2ndWon', 'B_SvGms', 'B_bpSaved', 'B_bpFaced']


### Antes y después

In [15]:
def perfil(d):
    mano_ok = d["A_hand"].isin(["R", "L"]) & d["B_hand"].isin(["R", "L"])
    return pd.Series({
        "Partidos": len(d),
        "Walkovers": int((d["score"].map(tipo_final) == "walkover").sum()),
        "Superficie Carpet": int((d["superficie"] == "Carpet").sum()),
        "Sin superficie": int(d["superficie"].isna().sum()),
        "Sin ranking (A o B)": int((d["A_rank"].isna() | d["B_rank"].isna()).sum()),
        "Sin estatura (A o B)": int((d["A_ht"].isna() | d["B_ht"].isna()).sum()),
        "Sin edad (A o B)": int((d["A_age"].isna() | d["B_age"].isna()).sum()),
        "Mano desconocida (A o B)": int((~mano_ok).sum()),
        "Exactitud regla ranking (%)": round(exactitud_regla(d) * 100, 1),
    })

def perfil_despues(d):
    p = perfil(d)
    p["Sin ranking (A o B)"] = int(((d["A_sin_ranking"] == 1) | (d["B_sin_ranking"] == 1)).sum())
    p["Exactitud regla ranking (%)"] = round(exactitud_regla(
        d.assign(A_rank=d["A_rank"].where(d["A_sin_ranking"] == 0),
                 B_rank=d["B_rank"].where(d["B_sin_ranking"] == 0))) * 100, 1)
    return p

antes_despues = pd.DataFrame({"Antes": perfil(original), "Después": perfil_despues(tabla)})
antes_despues

,Antes,Después
Partidos,79299.0,78815.0
Walkovers,423.0,0.0
Superficie Carpet,2039.0,0.0
Sin superficie,53.0,0.0
Sin ranking (A o B),1864.0,1751.0
Sin estatura (A o B),3559.0,0.0
Sin edad (A o B),25.0,0.0
Mano desconocida (A o B),1163.0,0.0
Exactitud regla ranking (%),65.6,65.7


En la fila "Sin ranking", el número de **después** cuenta a los jugadores que quedaron marcados como no clasificados: ya no son datos faltantes, sino una categoría con significado (ranking = 2.000 y `sin_ranking` = 1).

In [16]:
print("Filas eliminadas por decisión:")
print(pd.DataFrame(bitacora, columns=["Decisión", "Filas"]).to_string(index=False))
print(f"\nTotal: {len(original):,} → {len(tabla):,} partidos "
      f"({(len(original) - len(tabla)) / len(original):.2%} eliminados)")
print(f"Columnas: {original.shape[1]} → {tabla.shape[1]}")
print(f"Balance de la etiqueta después de limpiar: gana_A = {tabla['gana_A'].mean():.3f}")

Filas eliminadas por decisión:
                             Decisión  Filas
 Duplicados o jugador contra sí mismo      2
                     Partido walkover    423
                 Partido sin marcador      0
                 Partido no terminado      9
Sin superficie (no se pudo completar)     50

Total: 79,299 → 78,815 partidos (0.61% eliminados)
Columnas: 65 → 61
Balance de la etiqueta después de limpiar: gana_A = 0.497


### Guardar las tablas limpias

In [17]:
tabla.to_csv(os.path.join(CARPETA, "tabla_limpia.csv.gz"), index=False)
hist.drop(columns=["periodo"]).to_csv(os.path.join(CARPETA, "historial_limpio.csv.gz"), index=False)
print("Guardadas: tabla_limpia.csv.gz e historial_limpio.csv.gz")

Guardadas: tabla_limpia.csv.gz e historial_limpio.csv.gz


### Conclusión del paso 3
 
**Resultado:** de 79.299 a **78.815 partidos** (se eliminó el 0,6 %). La regla del ranking pasa de 65,6 % a **65,7 %**: la limpieza no cambió el problema, solo quitó lo que no se podía predecir.
 
| | Antes | Después |
|---|---|---|
| Partidos | 79.299 | 78.815 |
| Walkovers | 423 | 0 |
| Superficie Carpet | 2.039 | 0 |
| Sin superficie | 53 | 0 |
| Sin ranking (A o B) | 1.864 | 1.751 (marcados como no clasificados) |
| Sin estatura (A o B) | 3.559 | 0 |
| Sin edad (A o B) | 25 | 0 |
| Mano desconocida (A o B) | 1.163 | 0 |
| Exactitud regla ranking | 65,6 % | 65,7 % |
 
**Decisiones y su justificación**
- **Cómo terminó el partido:** el 96,5 % se jugó completo. Se eliminan los **423 walkovers** (el partido no se jugó) y los **9 sin terminar** (no se sabe quién ganó). Se **conservan los 2.364 retiros** (3 %) y las 20 descalificaciones: antes del partido nadie sabe que habrá un retiro, así que quitarlos haría la evaluación más fácil de lo que es en la realidad. La marca de retiro es información posterior al partido y nunca entra al modelo.
- **Duplicados:** un partido cargado dos veces; se dejó una sola copia.
- **Superficie:** Carpet (2.039 partidos, superficie que el ATP abandonó en 2009) se une con Hard. Los 53 sin superficie se completan con la superficie del mismo torneo en otros años; los que no se pudieron completar se eliminan.
- **Nivel "O":** son los 63 partidos de los **Juegos Olímpicos de París 2024**. Son partidos reales entre profesionales y se conservan.
- **Ranking:** de los unos 960 rankings que la unión encontró para jugadores sin ranking, la mitad tenía más de **417 días** y el 5 % más de **6 años**. Solo se usaron los de 60 días o menos (98). Los demás quedan como **no clasificados** (`sin_ranking = 1`, ranking 2.000, puntos 0). No es un dato perdido al azar, significa "peor que el último clasificado".
- **Estatura:** se compararon 5 métodos escondiendo estaturas conocidas (validación cruzada de 5 particiones):
 
| Método | Error medio |
|---|---|
| **Mediana por país** | **4,99 cm** |
| Mediana global | 5,08 cm |
| Mediana por década | 5,09 cm |
| Media global | 5,10 cm |
| Mediana por país y década | 5,33 cm |
 
  Gana la mediana por país, aunque por poco. El método más detallado (país y década) es el **peor**: al partir los datos en grupos muy pequeños, la mediana de cada grupo se vuelve ruidosa. Es sobreajuste, aplicado a la imputación.
- **Mano, edad, siembra y entrada:** la mano desconocida se imputa como diestro (la moda); la edad, desde la fecha de nacimiento; la siembra vacía = no sembrado; la entrada vacía = aceptación directa. La siembra y la entrada se publican con el cuadro, así que se conocen antes del partido.
- **Columnas con fuga:** el marcador, la duración, la marca de retiro y las 18 estadísticas del partido solo se conocen después de jugarlo. Quedan solo para construir el historial del paso 4.
 
**Siguiente paso:** crear las variables (Elo general y por superficie, forma reciente, historial directo, promedios de saque y cansancio), calculadas solo con partidos anteriores a cada fecha.